In [0]:
from pyspark.sql.functions import (
    col,
    row_number,
    monotonically_increasing_id
)
from pyspark.sql.window import Window

In [0]:
patients_silver = spark.table(
    "healthcare_catalog.silver.slv_patients"
)

In [0]:
#Create surrogate key:...
window_spec = Window.orderBy("patient_id")

dim_patient = patients_silver \
    .withColumn(
        "patient_key",
        row_number().over(window_spec)
    ) \
    .select(
        "patient_key",
        "patient_id",
        "first_name",
        "middle_name",
        "last_name",
        "birth_date",
        "death_date",
        "marital_status",
        "race",
        "ethnicity",
        "gender",
        "city",
        "state",
        "county",
        "zip_code",
        "latitude",
        "longitude",
        "healthcare_expenses",
        "healthcare_coverage",
        "income"
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_patient.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.dim_patient"
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.dim_patient
LIMIT 10;

patient_key,patient_id,first_name,middle_name,last_name,birth_date,death_date,marital_status,race,ethnicity,gender,city,state,county,zip_code,latitude,longitude,healthcare_expenses,healthcare_coverage,income
1,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,Jesica465,Leola603,Cormier289,1981-01-20,null,M,white,nonhispanic,F,Shirley,MASSACHUSETTS,Middlesex County,25017,1464.0,42.573316449173774,-71.63467207646659,402593.6,null
2,029c95e2-d252-d8d3-3626-7f347182d6d7,Clemencia569,Sina65,Gaylord332,1972-11-01,null,M,white,nonhispanic,F,Westford,MASSACHUSETTS,Middlesex County,null,0.0,42.54849073467201,-71.43706861726557,654939.59,null
3,031ce8e2-1443-01b4-de6b-61a4084a7166,Carmelo33,Jonathan639,Kirlin939,1981-01-08,2018-09-23,M,white,nonhispanic,M,Canton,MASSACHUSETTS,Norfolk County,null,0.0,42.17805313525782,-71.12722976935989,12869.13,null
4,06476266-59b1-ded7-4138-3c7543fd4981,Armand155,null,McGlynn426,1979-11-06,null,M,white,nonhispanic,M,Leominster,MASSACHUSETTS,Worcester County,25027,1420.0,42.55432794702615,-71.77509056604232,404615.87,null
5,09c573e2-a45d-aa5e-6a17-3c682155e6e3,Titus37,null,Langosh790,2008-02-12,null,null,white,nonhispanic,M,Medford,MASSACHUSETTS,Middlesex County,25017,2145.0,42.403159363884015,-71.07593516287898,56044.72,null
6,0b786670-17af-32e1-b2d2-f77c33874b30,Angel97,Elvin140,Mraz590,1964-06-23,2023-08-22,M,black,nonhispanic,M,Boston,MASSACHUSETTS,Suffolk County,25025,2131.0,42.321170096555036,-71.09325448207512,245397.84,null
7,0f3d04a2-f42e-1429-acef-0becd0a0fcea,Leland44,null,King743,1930-11-06,null,W,white,nonhispanic,M,Belmont,MASSACHUSETTS,Middlesex County,25017,2478.0,42.34762397067977,-71.09521517642047,132002.26,null
8,1426480f-ccc7-2393-40bd-f92ccf2b74b2,Rogelio17,Prince887,Monahan736,1961-12-18,null,M,white,nonhispanic,M,Sudbury,MASSACHUSETTS,Middlesex County,null,0.0,42.38283767153984,-71.4690433785207,412854.29,null
9,197ec76f-0730-2587-9ff9-a0f60104a782,Reynalda736,Janett802,Leuschke194,2007-06-28,null,null,white,nonhispanic,F,Norton Center,MASSACHUSETTS,Bristol County,null,0.0,42.02075454972385,-71.149831353573,47722.45,null
10,1be83f06-48ef-7bac-7097-b9e0644aeaf8,Darren774,Jarrett354,Rogahn59,1979-11-06,2026-05-03,D,white,nonhispanic,M,Leominster,MASSACHUSETTS,Worcester County,25027,1420.0,42.509982175526424,-71.73520860373362,197792.43,null


##Create dim_date

In [0]:
from pyspark.sql.functions import (
    sequence,
    to_date,
    explode,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    date_format
)

In [0]:
#generate dates ..
date_df = spark.sql("""
SELECT explode(
    sequence(
        to_date('2010-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )
) AS full_date
""")

In [0]:
dim_date = date_df \
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int")) \
    .withColumn("year", year("full_date")) \
    .withColumn("quarter", quarter("full_date")) \
    .withColumn("month", month("full_date")) \
    .withColumn("month_name", date_format("full_date", "MMMM")) \
    .withColumn("week_of_year", weekofyear("full_date")) \
    .withColumn("day", dayofmonth("full_date")) \
    .withColumn("day_of_week", dayofweek("full_date")) \
    .withColumn("day_name", date_format("full_date", "EEEE"))

In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.dim_date"
    )

In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.dim_date
LIMIT 10;

full_date,date_key,year,quarter,month,month_name,week_of_year,day,day_of_week,day_name
2010-01-01,20100101,2010,1,1,January,53,1,6,Friday
2010-01-02,20100102,2010,1,1,January,53,2,7,Saturday
2010-01-03,20100103,2010,1,1,January,53,3,1,Sunday
2010-01-04,20100104,2010,1,1,January,1,4,2,Monday
2010-01-05,20100105,2010,1,1,January,1,5,3,Tuesday
2010-01-06,20100106,2010,1,1,January,1,6,4,Wednesday
2010-01-07,20100107,2010,1,1,January,1,7,5,Thursday
2010-01-08,20100108,2010,1,1,January,1,8,6,Friday
2010-01-09,20100109,2010,1,1,January,1,9,7,Saturday
2010-01-10,20100110,2010,1,1,January,1,10,1,Sunday


In [0]:
#Create fact_encounter...
encounters_silver = spark.table(
    "healthcare_catalog.silver.slv_encounters"
)

dim_patient = spark.table(
    "healthcare_catalog.gold.dim_patient"
).select(
    "patient_key",
    "patient_id"
)  

In [0]:
encounters_silver = spark.table(
    "healthcare_catalog.silver.slv_encounters"
)

dim_patient = spark.table(
    "healthcare_catalog.gold.dim_patient"
).select(
    "patient_key",
    "patient_id"
)

In [0]:
fact_encounter = encounters_silver.join(
    dim_patient,
    on="patient_id",
    how="inner"
)

In [0]:
fact_encounter = fact_encounter.select(
    "encounter_id",
    "patient_key",
    "patient_id",
    "start_datetime",
    "end_datetime",
    "organization_id",
    "provider_id",
    "payer_id",
    "encounter_class",
    "encounter_code",
    "encounter_description",
    "base_encounter_cost",
    "total_claim_cost",
    "payer_coverage",
    "reason_code",
    "reason_description"
)

In [0]:
fact_encounter.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.fact_encounter"
    )

In [0]:
%sql
SELECT *
FROM healthcare_catalog.gold.fact_encounter
LIMIT 10;

encounter_id,patient_key,patient_id,start_datetime,end_datetime,organization_id,provider_id,payer_id,encounter_class,encounter_code,encounter_description,base_encounter_cost,total_claim_cost,payer_coverage,reason_code,reason_description
8b9df7af-24b3-66ee-7fad-869b4430a8aa,60,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2018-05-27T03:41:03.000Z,2018-05-27T04:41:03.000Z,1ddd51d1-7297-3c3b-bad5-98aecd5e4a79,978b6103-bc5e-3c77-a066-133aa89bbbf1,734afbd6-4794-363b-9bc0-6a3981533ed5,emergency,183460006,Obstetric emergency hospital admission (procedure),146.18,4512.52,4512.52,72892002,Normal pregnancy (finding)
8b9df7af-24b3-66ee-4eeb-9468ea972e4f,60,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2018-07-08T03:41:03.000Z,2018-07-08T03:56:03.000Z,2836d9ac-c6a6-39d1-8f13-fdf0fc8928f4,51936775-54b2-3316-b84c-195ddefe763c,734afbd6-4794-363b-9bc0-6a3981533ed5,ambulatory,169762003,Postnatal visit (regime/therapy),142.58,1005.38,1005.38,null,null
8b9df7af-24b3-66ee-91d1-5f83576c9c1f,60,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2020-08-23T03:41:03.000Z,2020-08-23T03:56:03.000Z,2836d9ac-c6a6-39d1-8f13-fdf0fc8928f4,51936775-54b2-3316-b84c-195ddefe763c,734afbd6-4794-363b-9bc0-6a3981533ed5,ambulatory,424619006,Prenatal visit (regime/therapy),142.58,8314.75,8314.75,72892002,Normal pregnancy (finding)
8b9df7af-24b3-66ee-276d-b0d31e4b09ba,60,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2022-02-01T21:41:03.000Z,2022-02-01T21:56:03.000Z,2836d9ac-c6a6-39d1-8f13-fdf0fc8928f4,51936775-54b2-3316-b84c-195ddefe763c,df166300-5a78-3502-a46a-832842197811,ambulatory,185345009,Encounter for symptom (procedure),85.55,624.47,574.47,10509002,Acute bronchitis (disorder)
65cbad6a-eb9d-420e-5ffe-48701db7582e,45,65cbad6a-eb9d-420e-9bab-b4f9c6aab7db,2013-10-01T06:35:38.000Z,2013-10-01T07:50:49.000Z,e2eab39c-5195-34aa-b35f-e89e37cfed01,bd19ee55-7b5f-3edb-9d74-a279232ace7a,a735bf55-83e9-331a-899d-a82a60b9f60c,ambulatory,185347001,Encounter for problem (procedure),85.55,1379.75,1103.8,37320007,Loss of teeth (disorder)
8b9df7af-24b3-66ee-2790-1781792485b4,60,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2023-07-24T03:41:03.000Z,2023-07-24T03:56:03.000Z,2836d9ac-c6a6-39d1-8f13-fdf0fc8928f4,51936775-54b2-3316-b84c-195ddefe763c,df166300-5a78-3502-a46a-832842197811,ambulatory,424619006,Prenatal visit (regime/therapy),142.58,1443.68,1393.68,72892002,Normal pregnancy (finding)
8b9df7af-24b3-66ee-d2b9-bc48cbb4ab75,60,8b9df7af-24b3-66ee-afeb-2734ebcbd33c,2023-11-16T03:41:03.000Z,2023-11-16T03:56:03.000Z,2836d9ac-c6a6-39d1-8f13-fdf0fc8928f4,51936775-54b2-3316-b84c-195ddefe763c,df166300-5a78-3502-a46a-832842197811,ambulatory,424619006,Prenatal visit (regime/therapy),142.58,1443.68,1393.68,72892002,Normal pregnancy (finding)
9a3814bb-f5b0-05f3-4c97-bdf8295b8a0b,67,9a3814bb-f5b0-05f3-928e-fe6640d5aa2a,2020-08-22T02:15:31.000Z,2020-08-22T02:30:31.000Z,afc661fd-71d9-3fc3-adf2-3fc3b5e8a140,4f69f42f-d99c-3263-986f-9ef6450d0be9,0133f751-9229-3cfd-815f-b6d4979bdd6a,ambulatory,394701000,Asthma follow-up (regime/therapy),142.58,142.58,0.0,233678006,Childhood asthma (disorder)
65cbad6a-eb9d-420e-f440-94ba212e4d11,45,65cbad6a-eb9d-420e-9bab-b4f9c6aab7db,2022-11-15T06:35:38.000Z,2022-11-15T09:35:58.000Z,e2eab39c-5195-34aa-b35f-e89e37cfed01,bd19ee55-7b5f-3edb-9d74-a279232ace7a,a735bf55-83e9-331a-899d-a82a60b9f60c,ambulatory,185349003,Encounter for check up (procedure),85.55,3105.35,2484.28,103697008,Patient referral for dental care (procedure)
355cd402-8f45-353f-0d67-50a419bea3f6,26,355cd402-8f45-353f-013f-ecb627e85106,2022-04-02T22:56:44.000Z,2022-04-02T23:13:27.000Z,352f2e3b-0708-3eb4-9f7e-e73a685bf379,4ba67eac-a99d-34c8-9055-4cf0013c75cc,734afbd6-4794-363b-9bc0-6a3981533ed5,ambulatory,185349003,Encounter for check up (procedure),85.55,948.35,758.68,263172003,Fracture of mandible (disorder)


In [0]:
#Create fact_condition...
conditions_silver = spark.table(
    "healthcare_catalog.silver.slv_conditions"
)

In [0]:
#Join patient key:...
fact_condition = conditions_silver.join(
    dim_patient,
    on="patient_id",
    how="inner"
)

In [0]:
fact_condition = fact_condition.select(
    "patient_id",
    "patient_key",
    "encounter_id",
    "condition_start_date",
    "condition_end_date",
    "condition_code",
    "condition_description"
)

In [0]:
fact_condition.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.fact_condition"
    )

In [0]:
#Create fact_medication...
medications_silver = spark.table(
    "healthcare_catalog.silver.slv_medications"
)

fact_medication = medications_silver.join(
    dim_patient,
    on="patient_id",
    how="inner"
) 

In [0]:
fact_medication = fact_medication.select(
    "patient_id",
    "patient_key",
    "encounter_id",
    "payer_id",
    "medication_start_datetime",
    "medication_end_datetime",
    "medication_code",
    "medication_description",
    "base_cost",
    "payer_coverage",
    "dispenses",
    "total_cost",
    "reason_code",
    "reason_description"
)

In [0]:
fact_medication.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.fact_medication"
    )

#Create fact_observation

In [0]:
observations_silver = spark.table(
    "healthcare_catalog.silver.slv_observations"
)

fact_observation = observations_silver.join(
    dim_patient,
    on="patient_id",
    how="inner"
) 

In [0]:
fact_observation = fact_observation.select(
    "patient_id",
    "patient_key",
    "encounter_id",
    "observation_datetime",
    "category",
    "observation_code",
    "observation_description",
    "observation_value",
    "units",
    "value_type"
) 

In [0]:
fact_observation.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.fact_observation"
    ) 

#Create fact_procedure

In [0]:
procedures_silver = spark.table(
    "healthcare_catalog.silver.slv_procedures"
)

fact_procedure = procedures_silver.join(
    dim_patient,
    on="patient_id",
    how="inner"
)

In [0]:
fact_procedure = fact_procedure.select(
    "patient_id",
    "patient_key",
    "encounter_id",
    "procedure_start_datetime",
    "procedure_end_datetime",
    "procedure_code",
    "procedure_description",
    "base_cost",
    "reason_code",
    "reason_description"
)

In [0]:
fact_procedure.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.fact_procedure"
    )

#Create fact_immunization

In [0]:
immunizations_silver = spark.table(
    "healthcare_catalog.silver.slv_immunizations"
)

fact_immunization = immunizations_silver.join(
    dim_patient,
    on="patient_id",
    how="inner"
)

In [0]:
fact_immunization = fact_immunization.select(
    "patient_id",
    "patient_key",
    "encounter_id",
    "immunization_datetime",
    "immunization_code",
    "immunization_description",
    "base_cost"
)

In [0]:
fact_immunization.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.fact_immunization"
    )

In [0]:
%sql
SHOW TABLES IN healthcare_catalog.gold;

database,tableName,isTemporary
gold,dim_date,false
gold,dim_patient,false
gold,fact_condition,false
gold,fact_encounter,false
gold,fact_immunization,false
gold,fact_medication,false
gold,fact_observation,false
gold,fact_procedure,false


In [0]:
%sql
SELECT 'dim_patient' AS table_name,
       COUNT(*) AS records
FROM healthcare_catalog.gold.dim_patient

UNION ALL

SELECT 'dim_date',
       COUNT(*)
FROM healthcare_catalog.gold.dim_date

UNION ALL

SELECT 'fact_encounter',
       COUNT(*)
FROM healthcare_catalog.gold.fact_encounter

UNION ALL

SELECT 'fact_condition',
       COUNT(*)
FROM healthcare_catalog.gold.fact_condition

UNION ALL

SELECT 'fact_medication',
       COUNT(*)
FROM healthcare_catalog.gold.fact_medication

UNION ALL

SELECT 'fact_observation',
       COUNT(*)
FROM healthcare_catalog.gold.fact_observation

UNION ALL

SELECT 'fact_procedure',
       COUNT(*)
FROM healthcare_catalog.gold.fact_procedure

UNION ALL

SELECT 'fact_immunization',
       COUNT(*)
FROM healthcare_catalog.gold.fact_immunization;

table_name,records
dim_patient,108
dim_date,7670
fact_encounter,5571
fact_condition,3517
fact_medication,3850
fact_observation,68648
fact_procedure,15884
fact_immunization,1549
